## Group No: 36

## Group Member Names:
1. RAVI SAJJANAR - 2024DA04008
2. DEVATA SAI SUDHESH - 2024DA04009
3. BANDARU HAREESHA - 2024DA04089
4. BHAVYA ARORA - 2023DA04058

## Journal used for the implementation
**Journal title:** Attention Is All You Need

**Authors:** Vaswani, A., Shazeer, N., Parmar, N., Uszkoreit, J., Jones, L., Gomez, A.N., Kaiser, Ł. and Polosukhin, I.

**Journal Name:** Advances in Neural Information Processing Systems (NeurIPS)

**Year:** 2017

# 1. Import the required libraries

In [1]:
##---------Type the code below this line------------------##
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score, precision_score, recall_score, f1_score
import pandas as pd
import re
import string
import pickle
import warnings
warnings.filterwarnings('ignore')

print("TensorFlow version:", tf.__version__)
print("GPU Available:", tf.config.list_physical_devices('GPU'))

/Users/hareeshabandaru/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


TensorFlow version: 2.20.0
GPU Available: []


# 2. Data Acquisition

For the problem identified by you, students have to find the data source themselves from any data source.

Provide the URL of the data used.

Write Code for converting the above downloaded data into a form suitable for DL



In [2]:
##---------Type the code below this line------------------##

# Dataset Information:
# We are using the Multi30k English-German translation dataset
# URL: https://github.com/multi30k/dataset
# This is a small-scale dataset suitable for demonstration purposes
# It contains ~29,000 training, ~1,000 validation, and ~1,000 test samples

# Download and extract the dataset
import urllib.request
import gzip
import os

# Create directory for data
os.makedirs('data', exist_ok=True)

# Download Multi30k dataset files
base_url = "https://raw.githubusercontent.com/multi30k/dataset/master/data/task1/raw/"

files = [
    "train.en.gz",
    "train.de.gz",
    "val.en.gz",
    "val.de.gz",
    "test_2016_flickr.en.gz",
    "test_2016_flickr.de.gz"
]

for file in files:
    url = base_url + file
    filepath = os.path.join('data', file)
    if not os.path.exists(filepath):
        print(f"Downloading {file}...")
        urllib.request.urlretrieve(url, filepath)

# Extract and read the files
def read_gzip_file(filepath):
    with gzip.open(filepath, 'rt', encoding='utf-8') as f:
        return f.read().strip().split('\n')

# Load training data
train_en = read_gzip_file('data/train.en.gz')
train_de = read_gzip_file('data/train.de.gz')

# Load validation data
val_en = read_gzip_file('data/val.en.gz')
val_de = read_gzip_file('data/val.de.gz')

# Load test data
test_en = read_gzip_file('data/test_2016_flickr.en.gz')
test_de = read_gzip_file('data/test_2016_flickr.de.gz')

print(f"Training samples: {len(train_en)}")
print(f"Validation samples: {len(val_en)}")
print(f"Test samples: {len(test_en)}")
print("\nSample English sentence:", train_en[0])
print("Sample German sentence:", train_de[0])

Training samples: 29000
Validation samples: 1014
Test samples: 1000

Sample English sentence: Two young, White males are outside near many bushes.
Sample German sentence: Zwei junge weiße Männer sind im Freien in der Nähe vieler Büsche.


# 3. Data Preparation

Perform the data preprocessing that is required for the data that you have downloaded.


This stage depends on the dataset that is used.

In [3]:
##---------Type the code below this line------------------##

# Text preprocessing function
def preprocess_sentence(sentence):
    """
    Preprocess text by:
    1. Converting to lowercase
    2. Adding spaces around punctuation
    3. Removing extra spaces
    4. Adding start and end tokens
    """
    sentence = sentence.lower().strip()
    # Add space before punctuation
    sentence = re.sub(r"([?.!,¿])", r" \1 ", sentence)
    sentence = re.sub(r'[" "]+', " ", sentence)
    # Remove extra spaces
    sentence = sentence.strip()
    # Add start and end tokens
    sentence = '<start> ' + sentence + ' <end>'
    return sentence

# Preprocess all sentences
train_en_processed = [preprocess_sentence(sent) for sent in train_en]
train_de_processed = [preprocess_sentence(sent) for sent in train_de]
val_en_processed = [preprocess_sentence(sent) for sent in val_en]
val_de_processed = [preprocess_sentence(sent) for sent in val_de]
test_en_processed = [preprocess_sentence(sent) for sent in test_en]
test_de_processed = [preprocess_sentence(sent) for sent in test_de]

print("Preprocessed English:", train_en_processed[0])
print("Preprocessed German:", train_de_processed[0])

# Create tokenizers
MAX_VOCAB_SIZE = 10000
MAX_LENGTH = 40

# Tokenizer for English (source)
en_tokenizer = tf.keras.preprocessing.text.Tokenizer(
    num_words=MAX_VOCAB_SIZE,
    filters='',
    oov_token='<unk>'
)
en_tokenizer.fit_on_texts(train_en_processed)

# Tokenizer for German (target)
de_tokenizer = tf.keras.preprocessing.text.Tokenizer(
    num_words=MAX_VOCAB_SIZE,
    filters='',
    oov_token='<unk>'
)
de_tokenizer.fit_on_texts(train_de_processed)

# Convert text to sequences
train_en_seq = en_tokenizer.texts_to_sequences(train_en_processed)
train_de_seq = de_tokenizer.texts_to_sequences(train_de_processed)
val_en_seq = en_tokenizer.texts_to_sequences(val_en_processed)
val_de_seq = de_tokenizer.texts_to_sequences(val_de_processed)
test_en_seq = en_tokenizer.texts_to_sequences(test_en_processed)
test_de_seq = de_tokenizer.texts_to_sequences(test_de_processed)

# Pad sequences
train_en_padded = tf.keras.preprocessing.sequence.pad_sequences(
    train_en_seq, maxlen=MAX_LENGTH, padding='post'
)
train_de_padded = tf.keras.preprocessing.sequence.pad_sequences(
    train_de_seq, maxlen=MAX_LENGTH, padding='post'
)
val_en_padded = tf.keras.preprocessing.sequence.pad_sequences(
    val_en_seq, maxlen=MAX_LENGTH, padding='post'
)
val_de_padded = tf.keras.preprocessing.sequence.pad_sequences(
    val_de_seq, maxlen=MAX_LENGTH, padding='post'
)
test_en_padded = tf.keras.preprocessing.sequence.pad_sequences(
    test_en_seq, maxlen=MAX_LENGTH, padding='post'
)
test_de_padded = tf.keras.preprocessing.sequence.pad_sequences(
    test_de_seq, maxlen=MAX_LENGTH, padding='post'
)

print(f"\nEnglish vocabulary size: {len(en_tokenizer.word_index)}")
print(f"German vocabulary size: {len(de_tokenizer.word_index)}")
print(f"\nPadded sequence shape: {train_en_padded.shape}")

## Split the data into training set and testing set
##---------Type the code below this line------------------##
# Data is already split into train, validation, and test sets
# Training set: train_en_padded, train_de_padded
# Validation set: val_en_padded, val_de_padded
# Test set: test_en_padded, test_de_padded

## Identify the target variables.
##---------Type the code below this line------------------##
# For sequence-to-sequence translation:
# Input: English sentences (train_en_padded)
# Target: German sentences (train_de_padded)
# The decoder uses shifted versions of target for teacher forcing

# Create decoder input and output
# Decoder input: all tokens except the last one
# Decoder output: all tokens except the first one (shifted)
train_de_input = train_de_padded[:, :-1]
train_de_output = train_de_padded[:, 1:]

val_de_input = val_de_padded[:, :-1]
val_de_output = val_de_padded[:, 1:]

test_de_input = test_de_padded[:, :-1]
test_de_output = test_de_padded[:, 1:]

print(f"\nTraining encoder input shape: {train_en_padded.shape}")
print(f"Training decoder input shape: {train_de_input.shape}")
print(f"Training decoder output shape: {train_de_output.shape}")

Preprocessed English: <start> two young , white males are outside near many bushes . <end>
Preprocessed German: <start> zwei junge weiße männer sind im freien in der nähe vieler büsche . <end>

English vocabulary size: 10422
German vocabulary size: 18837

Padded sequence shape: (29000, 40)

Training encoder input shape: (29000, 40)
Training decoder input shape: (29000, 39)
Training decoder output shape: (29000, 39)


## 4. Deep Neural Network Architecture


## 4.1 Design the architecture that you will be using

* CNN / RNN / Transformer as per the journal referenced



In [4]:
##---------Type the code below this line------------------##

# Transformer Architecture Components

# 1. Positional Encoding
def get_positional_encoding(seq_len, d_model):
    """
    Generate positional encoding as described in the paper.
    PE(pos, 2i) = sin(pos / 10000^(2i/d_model))
    PE(pos, 2i+1) = cos(pos / 10000^(2i/d_model))
    """
    positions = np.arange(seq_len)[:, np.newaxis]
    dimensions = np.arange(d_model)[np.newaxis, :]
    angle_rates = 1 / np.power(10000, (2 * (dimensions // 2)) / np.float32(d_model))
    angle_rads = positions * angle_rates

    # Apply sin to even indices
    angle_rads[:, 0::2] = np.sin(angle_rads[:, 0::2])
    # Apply cos to odd indices
    angle_rads[:, 1::2] = np.cos(angle_rads[:, 1::2])

    pos_encoding = angle_rads[np.newaxis, ...]
    return tf.cast(pos_encoding, dtype=tf.float32)

# 2. Multi-Head Attention Layer
class MultiHeadAttention(layers.Layer):
    def __init__(self, d_model, num_heads):
        super(MultiHeadAttention, self).__init__()
        self.num_heads = num_heads
        self.d_model = d_model

        assert d_model % self.num_heads == 0

        self.depth = d_model // self.num_heads

        self.wq = layers.Dense(d_model)
        self.wk = layers.Dense(d_model)
        self.wv = layers.Dense(d_model)

        self.dense = layers.Dense(d_model)

    def split_heads(self, x, batch_size):
        x = tf.reshape(x, (batch_size, -1, self.num_heads, self.depth))
        return tf.transpose(x, perm=[0, 2, 1, 3])

    def call(self, v, k, q, mask):
        batch_size = tf.shape(q)[0]

        q = self.wq(q)
        k = self.wk(k)
        v = self.wv(v)

        q = self.split_heads(q, batch_size)
        k = self.split_heads(k, batch_size)
        v = self.split_heads(v, batch_size)

        # Scaled dot-product attention
        matmul_qk = tf.matmul(q, k, transpose_b=True)
        dk = tf.cast(tf.shape(k)[-1], tf.float32)
        scaled_attention_logits = matmul_qk / tf.math.sqrt(dk)

        if mask is not None:
            scaled_attention_logits += (mask * -1e9)

        attention_weights = tf.nn.softmax(scaled_attention_logits, axis=-1)
        output = tf.matmul(attention_weights, v)

        output = tf.transpose(output, perm=[0, 2, 1, 3])
        concat_attention = tf.reshape(output, (batch_size, -1, self.d_model))
        output = self.dense(concat_attention)

        return output, attention_weights

# 3. Feed Forward Network
def point_wise_feed_forward_network(d_model, dff):
    return tf.keras.Sequential([
        layers.Dense(dff, activation='relu'),
        layers.Dense(d_model)
    ])

# 4. Encoder Layer
class EncoderLayer(layers.Layer):
    def __init__(self, d_model, num_heads, dff, dropout_rate=0.1):
        super(EncoderLayer, self).__init__()

        self.mha = MultiHeadAttention(d_model, num_heads)
        self.ffn = point_wise_feed_forward_network(d_model, dff)

        self.layernorm1 = layers.LayerNormalization(epsilon=1e-6)
        self.layernorm2 = layers.LayerNormalization(epsilon=1e-6)

        self.dropout1 = layers.Dropout(dropout_rate)
        self.dropout2 = layers.Dropout(dropout_rate)

    def call(self, x, training, mask):
        attn_output, _ = self.mha(x, x, x, mask)
        attn_output = self.dropout1(attn_output, training=training)
        out1 = self.layernorm1(x + attn_output)

        ffn_output = self.ffn(out1)
        ffn_output = self.dropout2(ffn_output, training=training)
        out2 = self.layernorm2(out1 + ffn_output)

        return out2

# 5. Decoder Layer
class DecoderLayer(layers.Layer):
    def __init__(self, d_model, num_heads, dff, dropout_rate=0.1):
        super(DecoderLayer, self).__init__()

        self.mha1 = MultiHeadAttention(d_model, num_heads)
        self.mha2 = MultiHeadAttention(d_model, num_heads)

        self.ffn = point_wise_feed_forward_network(d_model, dff)

        self.layernorm1 = layers.LayerNormalization(epsilon=1e-6)
        self.layernorm2 = layers.LayerNormalization(epsilon=1e-6)
        self.layernorm3 = layers.LayerNormalization(epsilon=1e-6)

        self.dropout1 = layers.Dropout(dropout_rate)
        self.dropout2 = layers.Dropout(dropout_rate)
        self.dropout3 = layers.Dropout(dropout_rate)

    def call(self, x, enc_output, training, look_ahead_mask, padding_mask):
        attn1, attn_weights_block1 = self.mha1(x, x, x, look_ahead_mask)
        attn1 = self.dropout1(attn1, training=training)
        out1 = self.layernorm1(attn1 + x)

        attn2, attn_weights_block2 = self.mha2(
            enc_output, enc_output, out1, padding_mask)
        attn2 = self.dropout2(attn2, training=training)
        out2 = self.layernorm2(attn2 + out1)

        ffn_output = self.ffn(out2)
        ffn_output = self.dropout3(ffn_output, training=training)
        out3 = self.layernorm3(ffn_output + out2)

        return out3, attn_weights_block1, attn_weights_block2

# 6. Encoder
class Encoder(layers.Layer):
    def __init__(self, num_layers, d_model, num_heads, dff, input_vocab_size,
                 maximum_position_encoding, dropout_rate=0.1):
        super(Encoder, self).__init__()

        self.d_model = d_model
        self.num_layers = num_layers

        self.embedding = layers.Embedding(input_vocab_size, d_model)
        self.pos_encoding = get_positional_encoding(maximum_position_encoding, d_model)

        self.enc_layers = [EncoderLayer(d_model, num_heads, dff, dropout_rate)
                          for _ in range(num_layers)]

        self.dropout = layers.Dropout(dropout_rate)

    def call(self, x, training, mask):
        seq_len = tf.shape(x)[1]

        x = self.embedding(x)
        x *= tf.math.sqrt(tf.cast(self.d_model, tf.float32))
        x += self.pos_encoding[:, :seq_len, :]

        x = self.dropout(x, training=training)

        for i in range(self.num_layers):
            x = self.enc_layers[i](x, training, mask)

        return x

# 7. Decoder
class Decoder(layers.Layer):
    def __init__(self, num_layers, d_model, num_heads, dff, target_vocab_size,
                 maximum_position_encoding, dropout_rate=0.1):
        super(Decoder, self).__init__()

        self.d_model = d_model
        self.num_layers = num_layers

        self.embedding = layers.Embedding(target_vocab_size, d_model)
        self.pos_encoding = get_positional_encoding(maximum_position_encoding, d_model)

        self.dec_layers = [DecoderLayer(d_model, num_heads, dff, dropout_rate)
                          for _ in range(num_layers)]
        self.dropout = layers.Dropout(dropout_rate)

    def call(self, x, enc_output, training, look_ahead_mask, padding_mask):
        seq_len = tf.shape(x)[1]
        attention_weights = {}

        x = self.embedding(x)
        x *= tf.math.sqrt(tf.cast(self.d_model, tf.float32))
        x += self.pos_encoding[:, :seq_len, :]

        x = self.dropout(x, training=training)

        for i in range(self.num_layers):
            x, block1, block2 = self.dec_layers[i](x, enc_output, training,
                                                   look_ahead_mask, padding_mask)

            attention_weights[f'decoder_layer{i+1}_block1'] = block1
            attention_weights[f'decoder_layer{i+1}_block2'] = block2

        return x, attention_weights

# 8. Complete Transformer Model
class Transformer(tf.keras.Model):
    def __init__(self, num_layers, d_model, num_heads, dff, input_vocab_size,
                 target_vocab_size, pe_input, pe_target, dropout_rate=0.1):
        super(Transformer, self).__init__()

        self.encoder = Encoder(num_layers, d_model, num_heads, dff,
                              input_vocab_size, pe_input, dropout_rate)

        self.decoder = Decoder(num_layers, d_model, num_heads, dff,
                              target_vocab_size, pe_target, dropout_rate)

        self.final_layer = layers.Dense(target_vocab_size)

    def call(self, inputs, training):
        inp, tar = inputs

        enc_padding_mask = self.create_padding_mask(inp)
        dec_padding_mask = self.create_padding_mask(inp)
        look_ahead_mask = self.create_look_ahead_mask(tf.shape(tar)[1])
        dec_target_padding_mask = self.create_padding_mask(tar)
        combined_mask = tf.maximum(dec_target_padding_mask, look_ahead_mask)

        enc_output = self.encoder(inp, training, enc_padding_mask)
        dec_output, attention_weights = self.decoder(
            tar, enc_output, training, combined_mask, dec_padding_mask)

        final_output = self.final_layer(dec_output)

        return final_output

    def create_padding_mask(self, seq):
        seq = tf.cast(tf.math.equal(seq, 0), tf.float32)
        return seq[:, tf.newaxis, tf.newaxis, :]

    def create_look_ahead_mask(self, size):
        mask = 1 - tf.linalg.band_part(tf.ones((size, size)), -1, 0)
        return mask

# Model hyperparameters (smaller than original paper for faster training)
NUM_LAYERS = 2  # Original paper uses 6
D_MODEL = 128   # Original paper uses 512
NUM_HEADS = 4   # Original paper uses 8
DFF = 512       # Original paper uses 2048
DROPOUT_RATE = 0.1

INPUT_VOCAB_SIZE = min(MAX_VOCAB_SIZE, len(en_tokenizer.word_index) + 1)
TARGET_VOCAB_SIZE = min(MAX_VOCAB_SIZE, len(de_tokenizer.word_index) + 1)

# Create the model
transformer = Transformer(
    num_layers=NUM_LAYERS,
    d_model=D_MODEL,
    num_heads=NUM_HEADS,
    dff=DFF,
    input_vocab_size=INPUT_VOCAB_SIZE,
    target_vocab_size=TARGET_VOCAB_SIZE,
    pe_input=MAX_LENGTH,
    pe_target=MAX_LENGTH,
    dropout_rate=DROPOUT_RATE
)

print("Transformer model created successfully!")
print(f"Model configuration: {NUM_LAYERS} layers, {D_MODEL} d_model, {NUM_HEADS} heads")

Transformer model created successfully!
Model configuration: 2 layers, 128 d_model, 4 heads


## 4.2 DNN Report

Report the following and provide justification for the same.

* Number of layers
* Number of units in each layer
* Total number of trainable parameters



# 5. Training the model


In [5]:
##---------Type the code below this line------------------##

# Transformer Architecture Components

# 1. Positional Encoding
def get_positional_encoding(seq_len, d_model):
    """
    Generate positional encoding as described in the paper.
    PE(pos, 2i) = sin(pos / 10000^(2i/d_model))
    PE(pos, 2i+1) = cos(pos / 10000^(2i/d_model))
    """
    positions = np.arange(seq_len)[:, np.newaxis]
    dimensions = np.arange(d_model)[np.newaxis, :]
    angle_rates = 1 / np.power(10000, (2 * (dimensions // 2)) / np.float32(d_model))
    angle_rads = positions * angle_rates

    # Apply sin to even indices
    angle_rads[:, 0::2] = np.sin(angle_rads[:, 0::2])
    # Apply cos to odd indices
    angle_rads[:, 1::2] = np.cos(angle_rads[:, 1::2])

    pos_encoding = angle_rads[np.newaxis, ...]
    return tf.cast(pos_encoding, dtype=tf.float32)

# 2. Multi-Head Attention Layer
class MultiHeadAttention(layers.Layer):
    def __init__(self, d_model, num_heads):
        super(MultiHeadAttention, self).__init__()
        self.num_heads = num_heads
        self.d_model = d_model

        assert d_model % self.num_heads == 0

        self.depth = d_model // self.num_heads

        self.wq = layers.Dense(d_model)
        self.wk = layers.Dense(d_model)
        self.wv = layers.Dense(d_model)

        self.dense = layers.Dense(d_model)

    def split_heads(self, x, batch_size):
        x = tf.reshape(x, (batch_size, -1, self.num_heads, self.depth))
        return tf.transpose(x, perm=[0, 2, 1, 3])

    def call(self, v, k, q, mask):
        batch_size = tf.shape(q)[0]

        q = self.wq(q)
        k = self.wk(k)
        v = self.wv(v)

        q = self.split_heads(q, batch_size)
        k = self.split_heads(k, batch_size)
        v = self.split_heads(v, batch_size)

        # Scaled dot-product attention
        matmul_qk = tf.matmul(q, k, transpose_b=True)
        dk = tf.cast(tf.shape(k)[-1], tf.float32)
        scaled_attention_logits = matmul_qk / tf.math.sqrt(dk)

        if mask is not None:
            scaled_attention_logits += (mask * -1e9)

        attention_weights = tf.nn.softmax(scaled_attention_logits, axis=-1)
        output = tf.matmul(attention_weights, v)

        output = tf.transpose(output, perm=[0, 2, 1, 3])
        concat_attention = tf.reshape(output, (batch_size, -1, self.d_model))
        output = self.dense(concat_attention)

        return output, attention_weights

# 3. Feed Forward Network
def point_wise_feed_forward_network(d_model, dff):
    return tf.keras.Sequential([
        layers.Dense(dff, activation='relu'),
        layers.Dense(d_model)
    ])

# 4. Encoder Layer
class EncoderLayer(layers.Layer):
    def __init__(self, d_model, num_heads, dff, dropout_rate=0.1):
        super(EncoderLayer, self).__init__()

        self.mha = MultiHeadAttention(d_model, num_heads)
        self.ffn = point_wise_feed_forward_network(d_model, dff)

        self.layernorm1 = layers.LayerNormalization(epsilon=1e-6)
        self.layernorm2 = layers.LayerNormalization(epsilon=1e-6)

        self.dropout1 = layers.Dropout(dropout_rate)
        self.dropout2 = layers.Dropout(dropout_rate)

    def call(self, x, mask, training=False):
        attn_output, _ = self.mha(x, x, x, mask)
        attn_output = self.dropout1(attn_output, training=training)
        out1 = self.layernorm1(x + attn_output)

        ffn_output = self.ffn(out1)
        ffn_output = self.dropout2(ffn_output, training=training)
        out2 = self.layernorm2(out1 + ffn_output)

        return out2

# 5. Decoder Layer
class DecoderLayer(layers.Layer):
    def __init__(self, d_model, num_heads, dff, dropout_rate=0.1):
        super(DecoderLayer, self).__init__()

        self.mha1 = MultiHeadAttention(d_model, num_heads)
        self.mha2 = MultiHeadAttention(d_model, num_heads)

        self.ffn = point_wise_feed_forward_network(d_model, dff)

        self.layernorm1 = layers.LayerNormalization(epsilon=1e-6)
        self.layernorm2 = layers.LayerNormalization(epsilon=1e-6)
        self.layernorm3 = layers.LayerNormalization(epsilon=1e-6)

        self.dropout1 = layers.Dropout(dropout_rate)
        self.dropout2 = layers.Dropout(dropout_rate)
        self.dropout3 = layers.Dropout(dropout_rate)

    def call(self, x, enc_output, look_ahead_mask, padding_mask, training=False):
        attn1, attn_weights_block1 = self.mha1(x, x, x, look_ahead_mask)
        attn1 = self.dropout1(attn1, training=training)
        out1 = self.layernorm1(attn1 + x)

        attn2, attn_weights_block2 = self.mha2(
            enc_output, enc_output, out1, padding_mask)
        attn2 = self.dropout2(attn2, training=training)
        out2 = self.layernorm2(attn2 + out1)

        ffn_output = self.ffn(out2)
        ffn_output = self.dropout3(ffn_output, training=training)
        out3 = self.layernorm3(ffn_output + out2)

        return out3, attn_weights_block1, attn_weights_block2

# 6. Encoder
class Encoder(layers.Layer):
    def __init__(self, num_layers, d_model, num_heads, dff, input_vocab_size,
                 maximum_position_encoding, dropout_rate=0.1):
        super(Encoder, self).__init__()

        self.d_model = d_model
        self.num_layers = num_layers

        self.embedding = layers.Embedding(input_vocab_size, d_model)
        self.pos_encoding = get_positional_encoding(maximum_position_encoding, d_model)

        self.enc_layers = [EncoderLayer(d_model, num_heads, dff, dropout_rate)
                          for _ in range(num_layers)]

        self.dropout = layers.Dropout(dropout_rate)

    def call(self, x, mask, training=False):
        seq_len = tf.shape(x)[1]

        x = self.embedding(x)
        x *= tf.math.sqrt(tf.cast(self.d_model, tf.float32))
        x += self.pos_encoding[:, :seq_len, :]

        x = self.dropout(x, training=training)

        for i in range(self.num_layers):
            x = self.enc_layers[i](x, mask, training=training)

        return x

# 7. Decoder
class Decoder(layers.Layer):
    def __init__(self, num_layers, d_model, num_heads, dff, target_vocab_size,
                 maximum_position_encoding, dropout_rate=0.1):
        super(Decoder, self).__init__()

        self.d_model = d_model
        self.num_layers = num_layers

        self.embedding = layers.Embedding(target_vocab_size, d_model)
        self.pos_encoding = get_positional_encoding(maximum_position_encoding, d_model)

        self.dec_layers = [DecoderLayer(d_model, num_heads, dff, dropout_rate)
                          for _ in range(num_layers)]
        self.dropout = layers.Dropout(dropout_rate)

    def call(self, x, enc_output, look_ahead_mask, padding_mask, training=False):
        seq_len = tf.shape(x)[1]
        attention_weights = {}

        x = self.embedding(x)
        x *= tf.math.sqrt(tf.cast(self.d_model, tf.float32))
        x += self.pos_encoding[:, :seq_len, :]

        x = self.dropout(x, training=training)

        for i in range(self.num_layers):
            x, block1, block2 = self.dec_layers[i](x, enc_output, look_ahead_mask,
                                                   padding_mask, training=training)

            attention_weights[f'decoder_layer{i+1}_block1'] = block1
            attention_weights[f'decoder_layer{i+1}_block2'] = block2

        return x, attention_weights

# 8. Complete Transformer Model
class Transformer(tf.keras.Model):
    def __init__(self, num_layers, d_model, num_heads, dff, input_vocab_size,
                 target_vocab_size, pe_input, pe_target, dropout_rate=0.1):
        super(Transformer, self).__init__()

        self.encoder = Encoder(num_layers, d_model, num_heads, dff,
                              input_vocab_size, pe_input, dropout_rate)

        self.decoder = Decoder(num_layers, d_model, num_heads, dff,
                              target_vocab_size, pe_target, dropout_rate)

        self.final_layer = layers.Dense(target_vocab_size)

    def call(self, inputs, training=False):
        inp, tar = inputs

        enc_padding_mask = self.create_padding_mask(inp)
        dec_padding_mask = self.create_padding_mask(inp)
        look_ahead_mask = self.create_look_ahead_mask(tf.shape(tar)[1])
        dec_target_padding_mask = self.create_padding_mask(tar)
        combined_mask = tf.maximum(dec_target_padding_mask, look_ahead_mask)

        enc_output = self.encoder(inp, enc_padding_mask, training=training)
        dec_output, attention_weights = self.decoder(
            tar, enc_output, combined_mask, dec_padding_mask, training=training)

        final_output = self.final_layer(dec_output)

        return final_output

    def create_padding_mask(self, seq):
        seq = tf.cast(tf.math.equal(seq, 0), tf.float32)
        return seq[:, tf.newaxis, tf.newaxis, :]

    def create_look_ahead_mask(self, size):
        mask = 1 - tf.linalg.band_part(tf.ones((size, size)), -1, 0)
        return mask

# Model hyperparameters (smaller than original paper for faster training)
NUM_LAYERS = 2  # Original paper uses 6
D_MODEL = 128   # Original paper uses 512
NUM_HEADS = 4   # Original paper uses 8
DFF = 512       # Original paper uses 2048
DROPOUT_RATE = 0.1

INPUT_VOCAB_SIZE = min(MAX_VOCAB_SIZE, len(en_tokenizer.word_index) + 1)
TARGET_VOCAB_SIZE = min(MAX_VOCAB_SIZE, len(de_tokenizer.word_index) + 1)

# Create the model
transformer = Transformer(
    num_layers=NUM_LAYERS,
    d_model=D_MODEL,
    num_heads=NUM_HEADS,
    dff=DFF,
    input_vocab_size=INPUT_VOCAB_SIZE,
    target_vocab_size=TARGET_VOCAB_SIZE,
    pe_input=MAX_LENGTH,
    pe_target=MAX_LENGTH,
    dropout_rate=DROPOUT_RATE
)

print("Transformer model created successfully!")
print(f"Model configuration: {NUM_LAYERS} layers, {D_MODEL} d_model, {NUM_HEADS} heads")

Transformer model created successfully!
Model configuration: 2 layers, 128 d_model, 4 heads


In [6]:
# Configure the training, by using appropriate optimizers, regularizations and loss functions
##---------Type the code below this line------------------##

# Custom learning rate schedule (as per the paper)
class CustomSchedule(tf.keras.optimizers.schedules.LearningRateSchedule):
    def __init__(self, d_model, warmup_steps=4000):
        super(CustomSchedule, self).__init__()
        self.d_model = d_model
        self.d_model = tf.cast(self.d_model, tf.float32)
        self.warmup_steps = warmup_steps

    def __call__(self, step):
        step = tf.cast(step, tf.float32)
        arg1 = tf.math.rsqrt(step)
        arg2 = step * (self.warmup_steps ** -1.5)
        return tf.math.rsqrt(self.d_model) * tf.math.minimum(arg1, arg2)

# Loss function with masking (ignore padding tokens)
loss_object = tf.keras.losses.SparseCategoricalCrossentropy(
    from_logits=True, reduction='none')

def loss_function(real, pred):
    mask = tf.math.logical_not(tf.math.equal(real, 0))
    loss_ = loss_object(real, pred)

    mask = tf.cast(mask, dtype=loss_.dtype)
    loss_ *= mask

    return tf.reduce_sum(loss_) / tf.reduce_sum(mask)

def accuracy_function(real, pred):
    # Cast argmax result to int32 to match real's dtype
    pred_ids = tf.cast(tf.argmax(pred, axis=2), tf.int32)
    accuracies = tf.equal(real, pred_ids)
    mask = tf.math.logical_not(tf.math.equal(real, 0))
    accuracies = tf.math.logical_and(mask, accuracies)
    accuracies = tf.cast(accuracies, dtype=tf.float32)
    mask = tf.cast(mask, dtype=tf.float32)
    return tf.reduce_sum(accuracies) / tf.reduce_sum(mask)

# Learning rate schedule
learning_rate = CustomSchedule(D_MODEL)

# Optimizer (Adam as per the paper)
optimizer = tf.keras.optimizers.Adam(
    learning_rate, beta_1=0.9, beta_2=0.98, epsilon=1e-9)

# Metrics
train_loss = tf.keras.metrics.Mean(name='train_loss')
train_accuracy = tf.keras.metrics.Mean(name='train_accuracy')
val_loss = tf.keras.metrics.Mean(name='val_loss')
val_accuracy = tf.keras.metrics.Mean(name='val_accuracy')

# Training step
@tf.function
def train_step(inp, tar):
    tar_inp = tar[:, :-1]
    tar_real = tar[:, 1:]

    with tf.GradientTape() as tape:
        predictions = transformer([inp, tar_inp], training=True)
        loss = loss_function(tar_real, predictions)

    gradients = tape.gradient(loss, transformer.trainable_variables)
    optimizer.apply_gradients(zip(gradients, transformer.trainable_variables))

    train_loss(loss)
    train_accuracy(accuracy_function(tar_real, predictions))

# Validation step
@tf.function
def val_step(inp, tar):
    tar_inp = tar[:, :-1]
    tar_real = tar[:, 1:]

    predictions = transformer([inp, tar_inp], training=False)
    loss = loss_function(tar_real, predictions)

    val_loss(loss)
    val_accuracy(accuracy_function(tar_real, predictions))

# Create datasets
BATCH_SIZE = 64
BUFFER_SIZE = 20000

train_dataset = tf.data.Dataset.from_tensor_slices((
    train_en_padded, train_de_padded
)).shuffle(BUFFER_SIZE).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

val_dataset = tf.data.Dataset.from_tensor_slices((
    val_en_padded, val_de_padded
)).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

# Training loop
EPOCHS = 20

history = {
    'train_loss': [],
    'train_accuracy': [],
    'val_loss': [],
    'val_accuracy': []
}

print("Starting training...\n")

for epoch in range(EPOCHS):
    train_loss.reset_state()
    train_accuracy.reset_state()
    val_loss.reset_state()
    val_accuracy.reset_state()

    # Training
    for (batch, (inp, tar)) in enumerate(train_dataset):
        train_step(inp, tar)

        if batch % 50 == 0:
            print(f'Epoch {epoch + 1} Batch {batch} Loss {train_loss.result():.4f} Accuracy {train_accuracy.result():.4f}')

    # Validation
    for (inp, tar) in val_dataset:
        val_step(inp, tar)

    # Save history
    history['train_loss'].append(float(train_loss.result()))
    history['train_accuracy'].append(float(train_accuracy.result()))
    history['val_loss'].append(float(val_loss.result()))
    history['val_accuracy'].append(float(val_accuracy.result()))

    print(f'\nEpoch {epoch + 1}: Train Loss: {train_loss.result():.4f}, Train Acc: {train_accuracy.result():.4f}')
    print(f'Val Loss: {val_loss.result():.4f}, Val Acc: {val_accuracy.result():.4f}\n')
    print('-' * 70)

print("\nTraining completed!")

# Display total parameters
total_params = sum([tf.size(var).numpy() for var in transformer.trainable_variables])
print(f"\nTotal trainable parameters: {total_params:,}")

Starting training...

Epoch 1 Batch 0 Loss 9.2090 Accuracy 0.0012
Epoch 1 Batch 50 Loss 9.1561 Accuracy 0.0133
Epoch 1 Batch 100 Loss 9.0443 Accuracy 0.0443
Epoch 1 Batch 150 Loss 8.9038 Accuracy 0.0655
Epoch 1 Batch 200 Loss 8.7267 Accuracy 0.0915
Epoch 1 Batch 250 Loss 8.5122 Accuracy 0.1098
Epoch 1 Batch 300 Loss 8.2695 Accuracy 0.1219
Epoch 1 Batch 350 Loss 8.0128 Accuracy 0.1309
Epoch 1 Batch 400 Loss 7.7568 Accuracy 0.1377
Epoch 1 Batch 450 Loss 7.5171 Accuracy 0.1456


2026-02-06 10:33:35.415884: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
2026-02-06 10:33:38.350760: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence



Epoch 1: Train Loss: 7.5037, Train Acc: 0.1460
Val Loss: 5.3361, Val Acc: 0.2158

----------------------------------------------------------------------
Epoch 2 Batch 0 Loss 5.2875 Accuracy 0.2373
Epoch 2 Batch 50 Loss 5.2828 Accuracy 0.2446
Epoch 2 Batch 100 Loss 5.1525 Accuracy 0.2534
Epoch 2 Batch 150 Loss 5.0265 Accuracy 0.2589
Epoch 2 Batch 200 Loss 4.9135 Accuracy 0.2665
Epoch 2 Batch 250 Loss 4.8087 Accuracy 0.2765
Epoch 2 Batch 300 Loss 4.7158 Accuracy 0.2856
Epoch 2 Batch 350 Loss 4.6321 Accuracy 0.2950
Epoch 2 Batch 400 Loss 4.5491 Accuracy 0.3041
Epoch 2 Batch 450 Loss 4.4718 Accuracy 0.3125


2026-02-06 10:36:33.437854: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence



Epoch 2: Train Loss: 4.4664, Train Acc: 0.3131
Val Loss: 3.6199, Val Acc: 0.3964

----------------------------------------------------------------------
Epoch 3 Batch 0 Loss 3.7806 Accuracy 0.3860
Epoch 3 Batch 50 Loss 3.6588 Accuracy 0.3951
Epoch 3 Batch 100 Loss 3.6056 Accuracy 0.4037
Epoch 3 Batch 150 Loss 3.5756 Accuracy 0.4085
Epoch 3 Batch 200 Loss 3.5434 Accuracy 0.4137
Epoch 3 Batch 250 Loss 3.5106 Accuracy 0.4188
Epoch 3 Batch 300 Loss 3.4748 Accuracy 0.4242
Epoch 3 Batch 350 Loss 3.4406 Accuracy 0.4294
Epoch 3 Batch 400 Loss 3.4083 Accuracy 0.4342
Epoch 3 Batch 450 Loss 3.3749 Accuracy 0.4394

Epoch 3: Train Loss: 3.3732, Train Acc: 0.4396
Val Loss: 2.8419, Val Acc: 0.5061

----------------------------------------------------------------------
Epoch 4 Batch 0 Loss 2.9174 Accuracy 0.5017
Epoch 4 Batch 50 Loss 2.8798 Accuracy 0.5060
Epoch 4 Batch 100 Loss 2.8493 Accuracy 0.5104
Epoch 4 Batch 150 Loss 2.8463 Accuracy 0.5104
Epoch 4 Batch 200 Loss 2.8346 Accuracy 0.5120
Epoch 4 

...


Epoch 4: Train Loss: 2.7275, Train Acc: 0.5275
Val Loss: 2.3343, Val Acc: 0.5745

----------------------------------------------------------------------
Epoch 5 Batch 0 Loss 2.2424 Accuracy 0.5971
Epoch 5 Batch 50 Loss 2.3672 Accuracy 0.5762
Epoch 5 Batch 100 Loss 2.3375 Accuracy 0.5810
Epoch 5 Batch 150 Loss 2.3323 Accuracy 0.5810
Epoch 5 Batch 200 Loss 2.3271 Accuracy 0.5812
Epoch 5 Batch 250 Loss 2.3082 Accuracy 0.5830
Epoch 5 Batch 300 Loss 2.3028 Accuracy 0.5836
Epoch 5 Batch 350 Loss 2.2971 Accuracy 0.5843
Epoch 5 Batch 400 Loss 2.2846 Accuracy 0.5861
Epoch 5 Batch 450 Loss 2.2772 Accuracy 0.5871

Epoch 5: Train Loss: 2.2760, Train Acc: 0.5872
Val Loss: 2.0255, Val Acc: 0.6166

----------------------------------------------------------------------
Epoch 6 Batch 0 Loss 1.8549 Accuracy 0.6512
Epoch 6 Batch 50 Loss 1.9713 Accuracy 0.6276
Epoch 6 Batch 100 Loss 1.9502 Accuracy 0.6316
Epoch 6 Batch 150 Loss 1.9612 Accuracy 0.6282
Epoch 6 Batch 200 Loss 1.9642 Accuracy 0.6276
Epoch 6 

2026-02-06 10:50:02.595861: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence



Epoch 8: Train Loss: 1.5614, Train Acc: 0.6784
Val Loss: 1.6746, Val Acc: 0.6636

----------------------------------------------------------------------
Epoch 9 Batch 0 Loss 1.3763 Accuracy 0.6961
Epoch 9 Batch 50 Loss 1.3291 Accuracy 0.7093
Epoch 9 Batch 100 Loss 1.3468 Accuracy 0.7073
Epoch 9 Batch 150 Loss 1.3554 Accuracy 0.7062
Epoch 9 Batch 200 Loss 1.3749 Accuracy 0.7034
Epoch 9 Batch 250 Loss 1.3922 Accuracy 0.7005
Epoch 9 Batch 300 Loss 1.4048 Accuracy 0.6985
Epoch 9 Batch 350 Loss 1.4163 Accuracy 0.6961
Epoch 9 Batch 400 Loss 1.4275 Accuracy 0.6948
Epoch 9 Batch 450 Loss 1.4341 Accuracy 0.6941

Epoch 9: Train Loss: 1.4345, Train Acc: 0.6940
Val Loss: 1.6498, Val Acc: 0.6712

----------------------------------------------------------------------
Epoch 10 Batch 0 Loss 1.3575 Accuracy 0.7004
Epoch 10 Batch 50 Loss 1.1921 Accuracy 0.7313
Epoch 10 Batch 100 Loss 1.1905 Accuracy 0.7298
Epoch 10 Batch 150 Loss 1.2156 Accuracy 0.7260
Epoch 10 Batch 200 Loss 1.2367 Accuracy 0.7218
Epo

2026-02-06 11:14:12.624428: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence



Epoch 16: Train Loss: 0.7966, Train Acc: 0.8000
Val Loss: 1.7121, Val Acc: 0.6887

----------------------------------------------------------------------
Epoch 17 Batch 0 Loss 0.6730 Accuracy 0.8117
Epoch 17 Batch 50 Loss 0.6505 Accuracy 0.8310
Epoch 17 Batch 100 Loss 0.6649 Accuracy 0.8263
Epoch 17 Batch 150 Loss 0.6756 Accuracy 0.8240
Epoch 17 Batch 200 Loss 0.6907 Accuracy 0.8207
Epoch 17 Batch 250 Loss 0.7040 Accuracy 0.8177
Epoch 17 Batch 300 Loss 0.7167 Accuracy 0.8151
Epoch 17 Batch 350 Loss 0.7265 Accuracy 0.8133
Epoch 17 Batch 400 Loss 0.7363 Accuracy 0.8116
Epoch 17 Batch 450 Loss 0.7457 Accuracy 0.8100

Epoch 17: Train Loss: 0.7466, Train Acc: 0.8099
Val Loss: 1.6729, Val Acc: 0.6888

----------------------------------------------------------------------
Epoch 18 Batch 0 Loss 0.4857 Accuracy 0.8791
Epoch 18 Batch 50 Loss 0.5996 Accuracy 0.8398
Epoch 18 Batch 100 Loss 0.6175 Accuracy 0.8361
Epoch 18 Batch 150 Loss 0.6313 Accuracy 0.8337
Epoch 18 Batch 200 Loss 0.6448 Accurac

# 6. Test the model


In [7]:
##---------Type the code below this line------------------##

# Test the model on test set
test_dataset = tf.data.Dataset.from_tensor_slices((
    test_en_padded, test_de_padded
)).batch(BATCH_SIZE)

test_loss = tf.keras.metrics.Mean(name='test_loss')
test_accuracy = tf.keras.metrics.Mean(name='test_accuracy')

print("Evaluating on test set...\n")

# Collect predictions for confusion matrix
all_predictions = []
all_targets = []

for (inp, tar) in test_dataset:
    tar_inp = tar[:, :-1]
    tar_real = tar[:, 1:]

    predictions = transformer([inp, tar_inp], training=False)
    loss = loss_function(tar_real, predictions)

    test_loss(loss)
    test_accuracy(accuracy_function(tar_real, predictions))

    # Collect predictions (token-level)
    pred_tokens = tf.argmax(predictions, axis=2).numpy()
    target_tokens = tar_real.numpy()

    # Flatten and filter out padding
    for i in range(len(pred_tokens)):
        mask = target_tokens[i] != 0
        all_predictions.extend(pred_tokens[i][mask].tolist())
        all_targets.extend(target_tokens[i][mask].tolist())

print(f"Test Loss: {test_loss.result():.4f}")
print(f"Test Accuracy: {test_accuracy.result():.4f}")

# Translation function for demonstration
def translate(sentence):
    # Preprocess input
    sentence = preprocess_sentence(sentence)
    sentence_seq = en_tokenizer.texts_to_sequences([sentence])
    sentence_padded = tf.keras.preprocessing.sequence.pad_sequences(
        sentence_seq, maxlen=MAX_LENGTH, padding='post'
    )

    encoder_input = tf.constant(sentence_padded)

    # Start with start token
    start_token = de_tokenizer.word_index['<start>']
    end_token = de_tokenizer.word_index['<end>']

    output = tf.TensorArray(dtype=tf.int32, size=0, dynamic_size=True)
    output = output.write(0, start_token)

    for i in range(MAX_LENGTH):
        dec_input = tf.transpose(output.stack())
        dec_input = tf.keras.preprocessing.sequence.pad_sequences(
            dec_input.numpy(), maxlen=MAX_LENGTH, padding='post'
        )
        dec_input = tf.constant(dec_input)

        predictions = transformer([encoder_input, dec_input], training=False)
        predictions = predictions[:, i, :]

        predicted_id = tf.cast(tf.argmax(predictions, axis=-1)[0], tf.int32)

        if predicted_id.numpy() == end_token:
            break

        output = output.write(i + 1, predicted_id)

    output = tf.transpose(output.stack())

    # Convert to text
    predicted_sentence = de_tokenizer.sequences_to_texts(output.numpy())[0]
    predicted_sentence = predicted_sentence.replace('<start>', '').replace('<end>', '').strip()

    return predicted_sentence

# Test translations
print("\nSample Translations:")
print("=" * 70)

test_sentences = [
    "A group of people stand in front of an igloo.",
    "A man in an orange shirt is playing guitar.",
    "Children are playing in the park."
]

for sent in test_sentences:
    translation = translate(sent)
    print(f"English: {sent}")
    print(f"German: {translation}")
    print("-" * 70)

Evaluating on test set...

Test Loss: 1.8133
Test Accuracy: 0.6844

Sample Translations:


ValueError: `sequences` must be a list of iterables. Found non-iterable: 2

# 7. Report the result

1. Plot the training and validation accuracy history.
2. Plot the training and validation loss history.
3. Report the testing accuracy and loss.
4. Show Confusion Matrix for testing dataset.
5. Report values for performance study metrics like accuracy, precision, recall, F1 Score.


In [ ]:
##---------Type the code below this line------------------##

# 1. Plot training and validation accuracy
plt.figure(figsize=(14, 5))

plt.subplot(1, 2, 1)
plt.plot(history['train_accuracy'], label='Training Accuracy', marker='o')
plt.plot(history['val_accuracy'], label='Validation Accuracy', marker='s')
plt.title('Model Accuracy Over Epochs', fontsize=14, fontweight='bold')
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Accuracy', fontsize=12)
plt.legend(loc='lower right')
plt.grid(True, alpha=0.3)

# 2. Plot training and validation loss
plt.subplot(1, 2, 2)
plt.plot(history['train_loss'], label='Training Loss', marker='o', color='orange')
plt.plot(history['val_loss'], label='Validation Loss', marker='s', color='red')
plt.title('Model Loss Over Epochs', fontsize=14, fontweight='bold')
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Loss', fontsize=12)
plt.legend(loc='upper right')
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/home/claude/training_history.png', dpi=300, bbox_inches='tight')
plt.show()

# 3. Report testing accuracy and loss
print("\n" + "=" * 70)
print("TEST SET PERFORMANCE")
print("=" * 70)
print(f"Test Loss: {test_loss.result():.4f}")
print(f"Test Accuracy: {test_accuracy.result():.4f} ({test_accuracy.result()*100:.2f}%)")
print("=" * 70)

# 4. Confusion Matrix (for top 10 most frequent tokens)
# Since we have ~10,000 classes, we'll show confusion matrix for top tokens
print("\nGenerating confusion matrix for top 20 most frequent tokens...")

# Limit to manageable number of classes
from collections import Counter
target_counts = Counter(all_targets)
top_tokens = [token for token, count in target_counts.most_common(20)]

# Filter predictions and targets to only include top tokens
filtered_preds = []
filtered_targets = []

for pred, target in zip(all_predictions, all_targets):
    if target in top_tokens:
        filtered_preds.append(pred if pred in top_tokens else -1)  # -1 for others
        filtered_targets.append(target)

# Create confusion matrix
cm = confusion_matrix(filtered_targets, filtered_preds, labels=top_tokens)

# Plot confusion matrix
plt.figure(figsize=(12, 10))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=True,
            xticklabels=top_tokens, yticklabels=top_tokens)
plt.title('Confusion Matrix - Top 20 Most Frequent Tokens', fontsize=14, fontweight='bold')
plt.ylabel('True Token ID', fontsize=12)
plt.xlabel('Predicted Token ID', fontsize=12)
plt.tight_layout()
plt.savefig('/home/claude/confusion_matrix.png', dpi=300, bbox_inches='tight')
plt.show()

# 5. Performance metrics
print("\n" + "=" * 70)
print("DETAILED PERFORMANCE METRICS (Token-Level)")
print("=" * 70)

# Calculate metrics for all tokens
accuracy = accuracy_score(all_targets, all_predictions)
precision = precision_score(all_targets, all_predictions, average='weighted', zero_division=0)
recall = recall_score(all_targets, all_predictions, average='weighted', zero_division=0)
f1 = f1_score(all_targets, all_predictions, average='weighted', zero_division=0)

print(f"Accuracy:  {accuracy:.4f} ({accuracy*100:.2f}%)")
print(f"Precision: {precision:.4f} ({precision*100:.2f}%)")
print(f"Recall:    {recall:.4f} ({recall*100:.2f}%)")
print(f"F1-Score:  {f1:.4f} ({f1*100:.2f}%)")
print("=" * 70)

# Create a summary table
metrics_df = pd.DataFrame({
    'Metric': ['Accuracy', 'Precision', 'Recall', 'F1-Score'],
    'Score': [accuracy, precision, recall, f1],
    'Percentage': [f"{accuracy*100:.2f}%", f"{precision*100:.2f}%",
                   f"{recall*100:.2f}%", f"{f1*100:.2f}%"]
})

print("\nPerformance Metrics Summary:")
print(metrics_df.to_string(index=False))

# Plot metrics
plt.figure(figsize=(10, 6))
metrics_names = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
metrics_values = [accuracy, precision, recall, f1]

bars = plt.bar(metrics_names, metrics_values, color=['#3498db', '#2ecc71', '#f39c12', '#e74c3c'])
plt.title('Performance Metrics on Test Set', fontsize=14, fontweight='bold')
plt.ylabel('Score', fontsize=12)
plt.ylim([0, 1])
plt.grid(True, alpha=0.3, axis='y')

# Add value labels on bars
for bar, value in zip(bars, metrics_values):
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2., height,
             f'{value:.4f}\n({value*100:.2f}%)',
             ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig('/home/claude/performance_metrics.png', dpi=300, bbox_inches='tight')
plt.show()

# Additional analysis
print("\n" + "=" * 70)
print("TRAINING SUMMARY")
print("=" * 70)
print(f"Total Epochs: {EPOCHS}")
print(f"Final Training Loss: {history['train_loss'][-1]:.4f}")
print(f"Final Training Accuracy: {history['train_accuracy'][-1]:.4f}")
print(f"Final Validation Loss: {history['val_loss'][-1]:.4f}")
print(f"Final Validation Accuracy: {history['val_accuracy'][-1]:.4f}")
print(f"\nBest Validation Accuracy: {max(history['val_accuracy']):.4f} (Epoch {history['val_accuracy'].index(max(history['val_accuracy'])) + 1})")
print(f"Best Validation Loss: {min(history['val_loss']):.4f} (Epoch {history['val_loss'].index(min(history['val_loss'])) + 1})")
print("=" * 70)

print("\n✓ All visualizations and metrics have been generated!")